# Module 04 · Unit 03
# Neural Network Topology

| | |
|---|---|
| **Estimated time** | 65–75 min |
| **Exercises** | Download PDF from the course repository |
| **Security thread** | Neural Network Architecture \[NN\] · Attack Graphs & Reachability \[AG\] |
| **Refresher** | Module 04 · Unit 02 — Directed Graphs and DAGs |

---

## Learning Objectives

By the end of this notebook you will be able to:

1. Represent a feedforward neural network as a **directed layered graph**
2. Model transformer **self-attention** as a complete weighted directed graph
3. Represent an **autograd computation graph** as a DAG and trace gradient flow
4. Model an **agentic MCP tool call network** as a directed graph with security constraints
5. Analyse what "removing a vertex" means in each context (ablation, compromise, segmentation)
6. Connect the graph-theoretic properties of neural architectures to their security implications


## 🔣 Symbol Primer

This unit uses all notation from Units 01–02. Three domain-specific symbols appear.

| Symbol | Name | Context | Meaning |
|---|---|---|---|
| $h_i^{(\ell)}$ | Hidden state | Neural networks | Activation of neuron $i$ in layer $\ell$ |
| $W^{(\ell)}$ | Weight matrix | Neural networks | Edge weights from layer $\ell$ to $\ell+1$ |
| $\alpha_{ij}$ | Attention weight | Transformers | How much position $i$ attends to position $j$ |
| $\partial L / \partial w$ | Gradient | Autograd | Rate of change of loss $L$ w.r.t. weight $w$ |

> **The payoff of Module 04.** You have spent two units learning graph vocabulary.
> This unit is where the vocabulary becomes genuinely useful: every major neural
> network architecture maps onto a specific graph structure you now know precisely.
> A feedforward net is a layered DAG. An RNN adds a cycle. A transformer's
> attention mechanism is a complete weighted directed graph. An autograd tape is
> a DAG of operations. Knowing the graph structure tells you what can go wrong,
> how gradients flow, and where security vulnerabilities live.

---


## 1 · Feedforward Networks as Layered DAGs

A **feedforward neural network** with layers $\ell = 0, 1, \ldots, L$ is a directed
graph where:

- Each **vertex** is a neuron — identified by its layer $\ell$ and position $i$
  within that layer: $(\ell, i)$
- Each **arc** connects every neuron in layer $\ell$ to every neuron in layer $\ell+1$
- **Edge weights** are the learned parameters $W^{(\ell)}_{ij}$

The graph is:
- **Directed** — information flows forward from inputs to outputs
- **Layered** — arcs only go from layer $\ell$ to layer $\ell+1$ (no skips in a plain MLP)
- **Acyclic** — there are no back-connections (making it a DAG)
- **Complete bipartite between adjacent layers** — every neuron in layer $\ell$
  connects to every neuron in layer $\ell+1$, forming $K_{n_\ell,\, n_{\ell+1}}$

The **forward pass** is a topological traversal of this DAG — layers are processed
in topological order (0, 1, 2, ..., L), exactly as Kahn's algorithm would order them.

### The Forward Pass as Graph Computation

For each layer $\ell$, the activation of neuron $j$ in layer $\ell + 1$ is:

$$h_j^{(\ell+1)} = \sigma\!\left(\sum_{i} W^{(\ell)}_{ij} \cdot h_i^{(\ell)} + b_j^{(\ell)}\right)$$

where $\sigma$ is a non-linear activation function (ReLU, sigmoid, etc.).

In graph language: **the activation of each vertex is a weighted sum of its
predecessors' activations.** The edges carry the weights; the vertices apply
the non-linearity.

### Node Removal — Ablation and Compromise

**Ablation** (ML research): force a neuron's activation to zero and observe the
effect on the network's output. This is vertex removal in the computation graph —
it is the same operation as removing a node from a network graph and observing
how reachability changes.

**Neuron compromise** (adversarial ML): an attacker who can influence the
activation of a specific neuron — via a backdoor trigger, for example — is
inserting a malicious arc into the computation graph. The downstream effect
propagates to all vertices reachable from the compromised neuron in the DAG.


## 2 · Transformer Self-Attention as a Complete Weighted Directed Graph

The **self-attention mechanism** in a transformer processes a sequence of $n$
tokens. For each pair of positions $(i, j)$, it computes an attention weight
$\alpha_{ij}$ — how much position $i$ "attends to" position $j$ when producing
its output representation.

Formally, this defines a **complete directed graph** on $n$ vertices (the token
positions):

- **Vertices:** $V = \{1, 2, \ldots, n\}$ — one per token position
- **Arcs:** $(i, j)$ for all $i, j$ — every position can attend to every other
- **Arc weight $\alpha_{ij}$:** the softmax-normalised attention score

$$\alpha_{ij} = \frac{\exp(q_i \cdot k_j / \sqrt{d_k})}{\sum_{j'} \exp(q_i \cdot k_{j'} / \sqrt{d_k})}$$

where $q_i$ is the query vector of position $i$ and $k_j$ is the key vector of
position $j$.

### Properties of the Attention Graph

- **Complete** — all $n^2$ arcs exist (in full attention; masked attention removes some)
- **Weighted** — weights $\alpha_{ij} \in [0, 1]$ and $\sum_j \alpha_{ij} = 1$ for each $i$
- **Not a DAG** in general — position $i$ can attend to itself and to positions
  both before and after it (bidirectional attention in BERT; masked in GPT)
- **Causal mask (GPT-style):** the upper triangle of weights is zeroed out —
  position $i$ can only attend to positions $j \leq i$. This makes the attention
  graph a **DAG** — no position attends to a future position.

### Security Connection

**Prompt injection** in a transformer exploits the attention mechanism. If an
attacker's injected text receives high attention weights from positions that represent
the original system prompt, the injected instructions can "override" the legitimate
ones. In graph terms: the injected tokens gain high-weight arcs pointing *toward*
the system-prompt tokens, allowing injected content to dominate the output.

Understanding attention as a weighted complete digraph makes this precise:
a prompt injection attack is an adversarial modification of the attention weight
matrix that changes which vertices dominate the output computation.


## 3 · The Autograd Computation Graph

**Automatic differentiation** (autograd) — the engine behind PyTorch and TensorFlow
backpropagation — records every mathematical operation as a DAG:

- **Vertices:** intermediate values (tensors) produced during the forward pass
- **Arcs:** $(u, v)$ means "value $u$ was used to compute value $v$"
- **Edge label:** the local derivative $\partial v / \partial u$

The **backward pass** traverses this DAG in *reverse topological order*,
accumulating gradients via the chain rule:

$$\frac{\partial L}{\partial u} = \sum_{v:\, (u,v) \in E} \frac{\partial L}{\partial v} \cdot \frac{\partial v}{\partial u}$$

In graph language: **the gradient of each vertex is the weighted sum of the
gradients of its successors, with weights given by the local derivatives.**
This is the reverse of the forward pass — same DAG, traversed backwards.

### Why Acyclicity is Non-Negotiable

If the computation graph had a cycle — say, output $y$ depends on itself — the
chain rule would produce an infinite recursion:

$$\frac{\partial L}{\partial y} = \cdots \cdot \frac{\partial y}{\partial y} = \cdots \cdot 1 + \cdots$$

Cycles make gradient computation undefined in the standard backpropagation framework.
This is why recurrent networks (RNNs) require **temporal unrolling** — the cyclic
computation graph is converted to a DAG by treating each time step as a separate
layer, creating a new set of vertices for each step.

### The Gradient Flow Security Connection

**Vanishing gradients** occur when the gradient signal — propagating backward
through the DAG — shrinks to near-zero by the time it reaches early layers.
In graph terms: the gradient paths from the loss (sink) to early weights (near
the sources) are long, and each arc multiplies by a small local derivative,
causing exponential decay. Deep network architectures (ResNets, LSTMs) are
graph-structural solutions to this problem — they add **skip connections**
(additional arcs from sources directly to later vertices) that provide short
gradient paths.


## 4 · Agentic AI — The MCP Tool Call Network

A **Model Context Protocol (MCP) agent** operates by calling tools exposed by
MCP servers. The interaction structure forms a directed graph:

- **Vertices:** the agent, MCP servers, and individual tools
- **Arcs:** $(\text{agent}, \text{server})$ for server connections;
  $(\text{server}, \text{tool})$ for tool exposure;
  $(\text{tool}, \text{agent})$ for results returned

The data flow during a task forms a **DAG of tool calls** — each tool's output
may feed into subsequent tool calls, but the overall workflow must terminate.

### Formal Security Properties (revisiting Module 02)

$$\forall \text{ tool call } t,\; t \in \text{workflow} \Rightarrow \text{granted}(\text{agent}, t)$$

*Every tool call the agent makes was explicitly granted.*

$$\text{workflow\_graph is a DAG}$$

*The tool call workflow terminates — no infinite loops.*

$$\forall \text{ server } s,\; \text{agent} \rightsquigarrow s \Rightarrow \text{authorised}(s)$$

*The agent only reaches authorised MCP servers.*

### Node Removal in the Agent Graph

**Removing a tool** (revoking a permission): removes a vertex and its incident arcs.
If the removed tool was required downstream, parts of the workflow become unreachable
from the task input — the workflow fails safely.

**Removing a server** (server compromise or shutdown): removes a server vertex
and all tools it exposes. Any tool that depended on those tools is now unreachable —
a controlled blast radius determined by the graph structure.

The graph tells you the blast radius *before* the incident.


---

## 🔐 Security Bridge &nbsp;·&nbsp; \[NN\] \[AG\]

| Neural architecture | Graph structure | Security / safety property |
|---|---|---|
| Feedforward MLP | Layered complete bipartite DAG | Forward pass terminates; gradient flows back |
| ResNet / skip connections | DAG with long-range arcs | Short gradient paths prevent vanishing gradient |
| RNN (unrolled) | DAG with temporal depth | Finite computation horizon |
| Transformer (causal) | Upper-triangular weighted DAG | No future information leakage |
| Transformer (full attention) | Complete weighted digraph | Bidirectional context; injection risk |
| Autograd tape | DAG of operations | Acyclicity ↔ gradient well-defined |
| MCP agent workflow | Directed tool call DAG | Terminates iff acyclic; blast radius = reachable tools |

**The unified view.** Whether you are auditing a neural network for adversarial
vulnerability, a dependency graph for supply chain risk, or an agent workflow for
safety guarantees, the question is always the same formal question:

1. What are the vertices?
2. What are the arcs?
3. Is the graph acyclic?
4. Which vertices are reachable from the threat source?
5. Which vertices are single points of failure?

Graph theory gives you the precise vocabulary to ask — and answer — those questions.

---


## Python: Visualization & Verification

**1 · Feedforward Network Graph** — build and visualise a small MLP as a layered
DAG; compute the forward pass through the graph structure; demonstrate what
ablating a neuron does to the output.

**2 · Transformer Attention Graph** — simulate scaled dot-product attention for
a short sequence; visualise the attention weight matrix as a weighted complete
digraph; compare full and causal (masked) attention.

**3 · MCP Agent Tool Call DAG** — model a complete agentic workflow with an
Agent node, two MCP servers, and six tools; visualise the hierarchy; verify
it is a DAG; simulate the effect of revoking a tool or compromising a server.


In [ ]:
import subprocess, sys

def install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

for pkg in ["numpy", "matplotlib", "sympy", "scipy", "networkx"]:
    install(pkg)
print("All packages installed.")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable
import networkx as nx

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({
    'figure.figsize': (9, 6), 'font.size': 11,
    'axes.titlesize': 13, 'axes.labelsize': 11,
    'lines.linewidth': 2, 'figure.dpi': 120,
})

TS_BLUE  = '#1e64b4'
TS_AMBER = '#c87814'
TS_GREEN = '#1e8c50'
TS_GRAY  = '#64646e'
TS_RED   = '#b41e1e'
TS_LIGHT = '#f5f7fa'

np.random.seed(42)
print('Setup complete.')


### 1 · Feedforward Network as a Layered DAG

We build a 3-layer MLP ($3 \to 4 \to 4 \to 2$) as a directed graph, compute
a forward pass through the graph structure (tracking activation propagation as
DAG traversal), and then demonstrate ablation: forcing one hidden neuron to zero
and observing how the output changes.


In [ ]:
# ── 1 · Feedforward Network as a Layered DAG ─────────────────────────────────

def relu(x): return np.maximum(0, x)

layer_sizes = [3, 4, 4, 2]   # input → hidden → hidden → output
L = len(layer_sizes) - 1

# Random weights (normally distributed, Xavier-ish)
weights = [np.random.randn(layer_sizes[l], layer_sizes[l+1]) * 0.5
           for l in range(L)]

# Build directed graph
MLP = nx.DiGraph()
node_ids = {}   # (layer, idx) → node name
for l, size in enumerate(layer_sizes):
    for i in range(size):
        name = f'L{l}_N{i}'
        node_ids[(l, i)] = name
        MLP.add_node(name, layer=l, idx=i)

for l in range(L):
    for i in range(layer_sizes[l]):
        for j in range(layer_sizes[l+1]):
            w = weights[l][i, j]
            MLP.add_edge(node_ids[(l, i)], node_ids[(l+1, j)], weight=w)

print(f"MLP Graph: |V|={MLP.number_of_nodes()}, |E|={MLP.number_of_edges()}")
print(f"Is DAG: {nx.is_directed_acyclic_graph(MLP)} ✓")

# ── Forward pass through the DAG ───────────────────────────────────────────────
x = np.array([0.8, -0.3, 0.5])   # sample input
activations = {}

topo = list(nx.topological_sort(MLP))
for node in topo:
    data  = MLP.nodes[node]
    layer = data['layer']
    idx   = data['idx']
    if layer == 0:
        activations[node] = x[idx]
    else:
        pre_act = sum(activations[pred] * MLP[pred][node]['weight']
                      for pred in MLP.predecessors(node))
        activations[node] = pre_act if layer == L else relu(pre_act)

output = np.array([activations[node_ids[(L, j)]] for j in range(layer_sizes[L])])
print(f"\nForward pass: input={x}  →  output={np.round(output, 4)}")

# ── Ablation: zero out neuron L1_N2 ───────────────────────────────────────────
ablated_activations = dict(activations)
ablated_node = 'L1_N2'
ablated_activations[ablated_node] = 0.0

for node in topo:
    data = MLP.nodes[node]
    layer, idx = data['layer'], data['idx']
    if layer > 0:
        pre_act = sum(ablated_activations.get(pred, activations[pred])
                      * MLP[pred][node]['weight']
                      for pred in MLP.predecessors(node))
        ablated_activations[node] = (pre_act if layer == L else relu(pre_act))

ablated_output = np.array([ablated_activations[node_ids[(L, j)]]
                            for j in range(layer_sizes[L])])
print(f"After ablating {ablated_node}: output={np.round(ablated_output, 4)}")
print(f"Output delta:                  {np.round(ablated_output - output, 4)}")

# ── Visualise ──────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 6))

# Positions: layer on x, neurons on y
pos = {}
for l, size in enumerate(layer_sizes):
    for i in range(size):
        pos[node_ids[(l, i)]] = (l * 3, -(i - size/2))

layer_colors = [TS_GREEN, TS_BLUE, TS_BLUE, TS_RED]
node_colors  = [TS_AMBER if n == ablated_node
                else layer_colors[MLP.nodes[n]['layer']]
                for n in MLP.nodes()]

# Draw edges coloured by weight sign
for u, v, data in MLP.edges(data=True):
    w     = data['weight']
    color = TS_BLUE if w > 0 else TS_RED
    alpha = min(0.8, abs(w) + 0.15)
    ax.annotate('', xy=pos[v], xytext=pos[u],
                arrowprops=dict(arrowstyle='->', color=color,
                                lw=max(0.5, abs(w)*2), alpha=alpha))

nx.draw_networkx_nodes(MLP, pos, ax=ax, node_color=node_colors,
                       node_size=700, alpha=0.95)
nx.draw_networkx_labels(MLP, pos, ax=ax, font_size=7,
                        font_color='white', font_weight='bold')

ax.text(0*3, -max(layer_sizes)/2 - 1.0, 'Input\n(L0)',  ha='center', fontsize=9, color=TS_GREEN)
ax.text(1*3, -max(layer_sizes)/2 - 1.0, 'Hidden\n(L1)', ha='center', fontsize=9, color=TS_BLUE)
ax.text(2*3, -max(layer_sizes)/2 - 1.0, 'Hidden\n(L2)', ha='center', fontsize=9, color=TS_BLUE)
ax.text(3*3, -max(layer_sizes)/2 - 1.0, 'Output\n(L3)', ha='center', fontsize=9, color=TS_RED)

legend = [mpatches.Patch(color=TS_GREEN,  label='Input neurons'),
          mpatches.Patch(color=TS_BLUE,   label='Hidden neurons'),
          mpatches.Patch(color=TS_RED,    label='Output neurons'),
          mpatches.Patch(color=TS_AMBER,  label='Ablated neuron (L1_N2)')]
ax.legend(handles=legend, loc='upper left', fontsize=8)
ax.set_title(f'Feedforward MLP as a Layered DAG\n'
             f'Blue edges = positive weight | Red edges = negative weight | '
             f'Amber node = ablated',
             pad=10, fontsize=10)
ax.axis('off')
ax.set_facecolor('white')
fig.patch.set_facecolor('white')
plt.tight_layout()
plt.show()


**What do you see?**

The MLP is drawn exactly as the layered DAG it is — all arrows pointing left to
right, no back-connections, complete connections between adjacent layers.

- **Blue edges** carry positive weights (excitatory connections).
- **Red edges** carry negative weights (inhibitory connections).
- **Edge thickness** reflects the weight magnitude — thicker edges carry more signal.
- **The amber node** (L1_N2) is ablated: its activation is forced to zero.

The ablation output delta shows how much the output changed when that neuron was
silenced. Neurons whose ablation causes large output changes are the high-degree
nodes of the *computational influence graph* — the neurons worth monitoring for
adversarial backdoor attacks.

**The formal connection:** ablation is vertex removal from the DAG. The blast
radius of ablating neuron $v$ is the set of output neurons reachable from $v$
— which, in a fully connected network, is always all output neurons. The
*magnitude* of the effect depends on the edge weights, not just the topology.
This is why weighted graph analysis matters.


### 2 · Transformer Attention as a Weighted Complete Directed Graph

We simulate scaled dot-product attention for a 6-token sequence, visualise the
attention weight matrix as both a heatmap and a directed graph, and compare full
bidirectional attention (BERT-style) against causal masked attention (GPT-style).


In [ ]:
# ── 2 · Transformer Attention Graph ─────────────────────────────────────────

tokens   = ['[SYS]', 'You', 'are', 'helpful', '[INJ]', 'Ignore']
n        = len(tokens)
d_k      = 8     # key dimension

# Simulate Q and K matrices
Q = np.random.randn(n, d_k)
K = np.random.randn(n, d_k)

# Compute raw attention scores
scores = Q @ K.T / np.sqrt(d_k)   # shape: (n, n)

# ── Full attention (softmax over all positions) ────────────────────────────────
def softmax_rows(X):
    X_exp = np.exp(X - X.max(axis=1, keepdims=True))
    return X_exp / X_exp.sum(axis=1, keepdims=True)

attn_full   = softmax_rows(scores)

# ── Causal (GPT-style) masked attention ───────────────────────────────────────
mask        = np.tril(np.ones((n, n)))  # lower triangle
scores_mask = np.where(mask, scores, -1e9)
attn_causal = softmax_rows(scores_mask)

# ── Plot: heatmaps ────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, attn, title in [
    (axes[0], attn_full,   'Full Attention (BERT-style)
Complete directed graph'),
    (axes[1], attn_causal, 'Causal Masked Attention (GPT-style)
Upper-triangular DAG'),
]:
    im = ax.imshow(attn, cmap='Blues', vmin=0, vmax=attn.max())
    for i in range(n):
        for j in range(n):
            val = attn[i, j]
            ax.text(j, i, f'{val:.2f}', ha='center', va='center',
                    fontsize=8, color='white' if val > 0.3 else TS_GRAY)

    ax.set_xticks(range(n)); ax.set_xticklabels(tokens, fontsize=9, rotation=30, ha='right')
    ax.set_yticks(range(n)); ax.set_yticklabels(tokens, fontsize=9)
    ax.set_xlabel('Key positions (attended to)', fontsize=9)
    ax.set_ylabel('Query positions (attending)', fontsize=9)
    ax.set_title(title, fontsize=10, fontweight='bold', pad=8)
    plt.colorbar(im, ax=ax, fraction=0.046)

plt.suptitle('Attention Weight Matrices — α[i,j] = how much position i attends to j',
             fontsize=11, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# ── Highlight the injection token's attention ──────────────────────────────────
inj_idx = tokens.index('[INJ]')
print("Attention weights FROM the [INJ] token to all others (full attention):")
for j, tok in enumerate(tokens):
    bar = '█' * int(attn_full[inj_idx, j] * 30)
    print(f"  {tok:<10}  {attn_full[inj_idx, j]:.3f}  {bar}")

print(f"\nSecurity note: if [INJ] receives high attention FROM [SYS] or other")
print(f"  positions, injected content can dominate the output representation.")
sys_idx = tokens.index('[SYS]')
print(f"  Attention from [SYS] to [INJ]: {attn_full[sys_idx, inj_idx]:.3f}")
print(f"  (In a real attack, adversarial text is crafted to maximise this weight)")


**What do you see?**

The **full attention** heatmap has no zero cells — every position can attend to
every other. The attention weights in each row sum to 1 (softmax normalisation).
The [INJ] and 'Ignore' tokens can receive attention from the [SYS] system prompt
token — this is the mechanism prompt injection exploits.

The **causal mask** heatmap has zeros in the upper triangle — position $i$ can
only attend to positions $j \leq i$. In graph terms, this removes all arcs
$(i, j)$ with $j > i$, converting the complete digraph into a **DAG** (upper
triangular, no cycles). GPT-style generation is a DAG computation; BERT-style
masked language modelling is a complete digraph computation.

**The prompt injection connection.** In the full attention case, if an attacker
can embed the [INJ] token such that $\alpha_{\text{SYS},\text{INJ}}$ is large —
meaning the system prompt token attends heavily to the injection token — the
model's representation of the system prompt is polluted by the injected content.
The attention weight matrix is the attack surface; the graph structure tells you
which arcs to attack.


### 3 · MCP Agent Tool Call Network

We model a complete agentic AI deployment with an agent, two MCP servers, and
six tools. The graph captures both the structural hierarchy (agent → server → tool)
and the data dependencies between tool calls during a specific task.

We then simulate two security scenarios:
- **Tool revocation:** remove a tool and identify which downstream capabilities
  become unreachable
- **Server compromise:** remove an entire server and compute the blast radius


In [ ]:
# ── 3 · MCP Agent Tool Call Network ─────────────────────────────────────────

# Nodes: agent, servers, tools
node_types = {
    'agent':           'agent',
    'mcp_security':    'server',
    'mcp_filesystem':  'server',
    'list_users':      'tool',
    'check_mfa':       'tool',
    'read_audit_log':  'tool',
    'write_report':    'tool',
    'list_files':      'tool',
    'read_config':     'tool',
}

# Structural graph: agent → servers → tools
structural_edges = [
    ('agent',          'mcp_security'),
    ('agent',          'mcp_filesystem'),
    ('mcp_security',   'list_users'),
    ('mcp_security',   'check_mfa'),
    ('mcp_security',   'read_audit_log'),
    ('mcp_filesystem', 'write_report'),
    ('mcp_filesystem', 'list_files'),
    ('mcp_filesystem', 'read_config'),
]

# Data dependency edges during a specific audit task
dependency_edges = [
    ('list_users',     'check_mfa'),         # need users to check MFA status
    ('check_mfa',      'write_report'),       # MFA findings → report
    ('read_audit_log', 'write_report'),       # audit findings → report
    ('read_config',    'read_audit_log'),     # config tells where audit log is
    ('list_files',     'read_config'),        # list files to find config
]

MCP = nx.DiGraph()
MCP.add_nodes_from(node_types.keys())
MCP.add_edges_from(structural_edges)
MCP.add_edges_from(dependency_edges)

print(f"MCP Agent Graph: |V|={MCP.number_of_nodes()}, |E|={MCP.number_of_edges()}")
print(f"Is DAG: {nx.is_directed_acyclic_graph(MCP)}")

# Topological execution order
topo = list(nx.topological_sort(MCP))
print(f"\nExecution order (topological sort):")
for i, node in enumerate(topo):
    role = node_types[node]
    print(f"  {i+1}. [{role:>6}]  {node}")

# ── Blast radius analysis: remove mcp_security ────────────────────────────────
MCP_no_security = MCP.copy()
MCP_no_security.remove_node('mcp_security')

# What is now unreachable from agent?
reachable_before = set(nx.descendants(MCP, 'agent')) | {'agent'}
reachable_after  = set(nx.descendants(MCP_no_security, 'agent')) | {'agent'}
lost = reachable_before - reachable_after

print(f"\nBlast radius of removing 'mcp_security':")
print(f"  Reachable before: {sorted(reachable_before)}")
print(f"  Reachable after:  {sorted(reachable_after)}")
print(f"  Lost capabilities: {sorted(lost)}")

# ── Visualise ──────────────────────────────────────────────────────────────────
type_pal = {'agent': '#8e44ad', 'server': TS_BLUE, 'tool': TS_GREEN}
edge_pal = {'structural': TS_GRAY, 'dependency': TS_AMBER}

fig, axes = plt.subplots(1, 2, figsize=(14, 7))

for ax, graph, title, removed in [
    (axes[0], MCP,              'Full MCP Agent Graph',           None),
    (axes[1], MCP_no_security,  'After mcp_security compromised', 'mcp_security'),
]:
    pos = nx.spring_layout(graph, seed=5, k=3.5)
    colors = [type_pal[node_types[v]] for v in graph.nodes()]

    # Draw structural vs dependency edges differently
    struct = [(u,v) for u,v in graph.edges() if (u,v) in structural_edges]
    deps   = [(u,v) for u,v in graph.edges() if (u,v) in dependency_edges]

    nx.draw_networkx_edges(graph, pos, ax=ax, edgelist=struct,
                           edge_color=TS_GRAY, alpha=0.5, arrows=True,
                           arrowsize=16, width=2,
                           connectionstyle='arc3,rad=0.05')
    nx.draw_networkx_edges(graph, pos, ax=ax, edgelist=deps,
                           edge_color=TS_AMBER, alpha=0.7, arrows=True,
                           arrowsize=16, width=1.5, style='dashed',
                           connectionstyle='arc3,rad=0.15')
    nx.draw_networkx_nodes(graph, pos, ax=ax, node_color=colors,
                           node_size=900, alpha=0.92)
    nx.draw_networkx_labels(graph, pos, ax=ax, font_size=7,
                            font_color='white', font_weight='bold')

    # Mark lost nodes in red outline if in the compromised graph
    if removed:
        lost_in_graph = [v for v in graph.nodes() if v in lost]
        if lost_in_graph:
            nx.draw_networkx_nodes(graph, pos, ax=ax, nodelist=lost_in_graph,
                                   node_color=[TS_RED]*len(lost_in_graph),
                                   node_size=950, alpha=0.5)

    ax.set_title(title, fontsize=10, fontweight='bold', pad=8)
    ax.axis('off')
    ax.set_facecolor('white')

legend_elements = [
    mpatches.Patch(color='#8e44ad', label='Agent'),
    mpatches.Patch(color=TS_BLUE,   label='MCP Server'),
    mpatches.Patch(color=TS_GREEN,  label='Tool'),
    mpatches.Patch(color=TS_RED,    label='Lost after compromise'),
    mpatches.Patch(color=TS_AMBER,  label='Data dependency (dashed)'),
]
fig.legend(handles=legend_elements, loc='lower center',
           ncol=5, fontsize=8, bbox_to_anchor=(0.5, -0.02))
fig.patch.set_facecolor('white')
plt.suptitle('MCP Agent Tool Call Network — Structure and Blast Radius',
             fontsize=12, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()


**What do you see?**

The full graph (left) shows two layers of structure:
- **Solid grey arcs** — structural connections (agent to servers, servers to tools)
- **Dashed amber arcs** — data dependencies (tool outputs feeding into other tools)

The topological sort gives the correct execution order: start with `list_files`
and `read_config` (no prerequisites), then work up through the dependency chain
to `write_report` (everything feeds into the final report).

After compromising `mcp_security` (right), the red nodes show the blast radius:
`list_users`, `check_mfa`, and `read_audit_log` all become unreachable — the
agent can no longer perform MFA checks or read audit logs. The report can still
be written (it loses some inputs), but critical security capabilities are gone.

**The graph tells you the blast radius before the incident.** This is the formal
security analysis that Agent Scrutiny-style frameworks apply: given the tool call
graph and a set of compromised or revoked nodes, compute the reachability change.
The mathematics is Module 04 graph theory; the application is real-time agent
security monitoring.


In [ ]:
# ── Extension Challenge ───────────────────────────────────────────────────────
#
# 1. GRADIENT FLOW PATHS
#    In the MLP DAG, implement a function that finds all directed paths
#    from a given hidden neuron to the output layer.
#    For each path, compute the product of |edge weights| along the path —
#    this approximates the gradient contribution from that neuron.
#    Which neurons have the strongest gradient paths to the output?
#    This is a simplified version of saliency analysis.
#
# 2. ATTENTION GRAPH CLUSTERING
#    For the full attention matrix, threshold it: keep only arcs where
#    alpha[i,j] > 0.2 (strong attention links).
#    Compute the connected components of the resulting undirected graph.
#    Which tokens cluster together? Does the [INJ] token cluster with
#    the system prompt or with the instruction tokens?
#
# 3. MULTI-AGENT GRAPH
#    Extend the MCP graph to two agents (agent_a, agent_b) sharing
#    the mcp_filesystem server but each having their own security server.
#    Add an agent-to-agent communication edge: agent_a can request
#    agent_b to run a task.
#    Is this still a DAG? If agent_b can also request agent_a, is it?
#    What security property does the DAG requirement enforce?

# Your work here:


---

## Summary

| Architecture | Graph structure | Key property |
|---|---|---|
| Feedforward MLP | Layered complete bipartite DAG | Topological forward pass; ablation = vertex removal |
| ResNet | DAG with skip connections | Short gradient paths |
| Causal transformer | Upper-triangular weighted DAG | No future leakage; DAG = finite horizon |
| Full transformer | Complete weighted directed graph | Bidirectional context; injection risk |
| Autograd tape | DAG of operations | Acyclicity = gradient well-defined |
| MCP agent workflow | Directed tool call DAG | Terminates; blast radius = reachable nodes |

---

## Module 04 Complete

| Unit | Content | Payoff |
|---|---|---|
| **01** | Graph definitions, degree, paths, connectivity, bipartite | The vocabulary for any networked system |
| **02** | Directed graphs, DAGs, topological sort | Dependency analysis; cycle detection; execution ordering |
| **03** | Neural network topology; attention; autograd; agent graphs | Every major AI architecture as a formal graph |

The three units build a single argument: **every system you care about — networks,
software, neural models, agent workflows — is a graph. The properties of that graph
determine what is reachable, what can fail, and what the blast radius looks like.**

---

## Up Next

**Module 05 — Graph Theory II: Algorithms**

We add algorithms to the structure. Breadth-first and depth-first search for
reachability. Dijkstra's algorithm for minimum-cost attack paths on weighted
graphs. And the formal analysis of how removing edges (zero-trust segmentation)
raises the minimum-cost lateral movement path — the mathematics of why network
segmentation works.

$\rightarrow$ `modules/module-05/unit-01-traversal-reachability.ipynb`
